# Prueba de acceso VNC · sin Playwright

Objetivo: verificar salud, autenticación, imagen completa, clic y teclado. Cada herramienta imprime su resultado JSON y las observaciones muestran la pantalla.

**Credenciales:** se cargan privadamente desde `.env.vnc`. No imprimir configuración ni objetos internos. **Publicación:** mantener esta copia sin outputs; guardar la ejecución en `.private/notebooks/`.

Este notebook valida herramientas para el futuro agente visual. Las decisiones autónomas y los subagentes se implementarán después. Las coordenadas de la demostración son específicas de la pantalla observada, no reglas del agente general.

In [ ]:
from pathlib import Path
import sys, json, warnings
from IPython.display import display
from dotenv import dotenv_values
import httpx
from cryptography.utils import CryptographyDeprecationWarning
warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/vnc_probe.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from vnc_probe import VNCProbe

config = dotenv_values(ROOT / ".env.vnc")
required = ("VNC_BASE_URL", "VNC_WS_URL", "VNC_PASSWORD")
missing = [key for key in required if not config.get(key)]
if missing:
    raise RuntimeError("Falta configuración: " + ", ".join(missing))
print(json.dumps({"command": "configuration", "ok": True, "configured_variables": list(required)}))

## 1. Salud del escritorio
Petición GET de lectura; no ejecutamos reset/check.

In [ ]:
try:
    async with httpx.AsyncClient(timeout=15) as http:
        response = await http.get(config["VNC_BASE_URL"].rstrip("/") + "/health")
    assert response.status_code == 200 and response.json().get("ok") is True
    print(json.dumps({"command": "health", "status": response.status_code, "ok": True}))
except Exception as exc:
    raise RuntimeError(f"Health failed ({type(exc).__name__}); destination hidden") from None

## 2. Conexión directa
AsyncVNC implementa RFB. El puente local convierte el transporte TCP a WebSocket binario; solo escucha en 127.0.0.1. La sesión tiene un límite de 180 segundos.

In [ ]:
probe = VNCProbe(config["VNC_WS_URL"], config["VNC_PASSWORD"], seconds=180)
await probe.connect()

## 3. Observar la pantalla completa
Enviamos Escape y observamos; su efecto depende de la app. La imagen proviene del framebuffer VNC, no de una ventana ni del DOM.

In [ ]:
await probe.press_key("Esc")
image = await probe.observe()
display(image)
probe.save_image(ROOT / ".private/evidencias/vnc-direct-initial.png")

## 4. Probar un clic de navegación

Revisa la imagen anterior. En la pantalla de bienvenida de SIGECOM observada, el menú **Archivo** está cerca de `(40, 48)`. Si la app cambió o hay un diálogo, ajusta las coordenadas y mantén `RUN_MENU_DEMO = False` hasta identificar un control de navegación seguro.

No usar esta celda para guardar, borrar o modificar registros.

In [ ]:
RUN_MENU_DEMO = True
MENU_X, MENU_Y = 40, 48
if RUN_MENU_DEMO:
    try:
        await probe.click(MENU_X, MENU_Y, hold_ms=180)
        display(await probe.observe())
        probe.save_image(ROOT / ".private/evidencias/vnc-direct-menu.png")
    except BaseException:
        await probe.disconnect()
        raise
else:
    print(json.dumps({"command": "click", "skipped": True}))

## 5. Mover el puntero y probar teclado
Alejamos el puntero del menú para evitar que el hover lo mantenga abierto y enviamos Tab para probar navegación de foco. En la prueba observada Escape no cerró el menú de esta app; no debemos asumir ese comportamiento. La imagen permite comprobar el resultado; una confirmación de transporte por sí sola no demuestra que la app procesó la tecla.

In [ ]:
try:
    await probe.move_pointer(600, 760)
    await probe.press_key("Tab")
    display(await probe.observe())
    probe.save_image(ROOT / ".private/evidencias/vnc-direct-final.png")
except BaseException:
    await probe.disconnect()
    raise

## 6. Imagen para el futuro modelo visual
El agente recibirá el PNG como imagen. Aquí solo mostramos formato y tamaño, nunca el base64 completo ni credenciales.

In [ ]:
image_payload = probe.image_for_model()
print(json.dumps({"command": "image_for_model", "mime_type": "image/png", "encoded_characters": len(image_payload), "ok": True}))
# Un adaptador del modelo visual consumirá image_payload + instrucción y
# devolverá una acción estructurada. No enviamos la imagen a un proveedor en esta prueba.

## 7. Desconectar y ver el historial
Cerrar siempre al terminar. La herramienta replace_text también existe, pero no la ejecutamos sin verificar el foco de un campo.

In [ ]:
await probe.disconnect()
print(json.dumps({"command": "summary", "events": len(probe.events), "connection_closed": True}))
display(probe.events)

## Qué demuestra esta prueba

- Acceso WSS y autenticación VNC.
- Reconstrucción de la pantalla y disponibilidad de imágenes para un modelo.
- Envío de clics prolongados y teclas, seguido de observación.
- Desconexión y registro sin secretos.

No prueba todavía interpretación por un modelo, edición de campos, exportación, éxito en E1–E5 ni cancelación de un agente completo. El límite de esta sesión es de laboratorio; la implementación productiva deberá probar carreras y respuestas tardías.